# Crypto.Net · 03 — Ethereum dan EVM

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

> **Aman dijalankan.** Notebook ini hanya membaca jaringan publik dan menandatangani secara offline; tidak pernah mem-broadcast transaksi. Kunci yang dibuat di sini adalah kunci demo — jangan pernah mengirim dana sungguhan ke sana.

Buka di VS Code dengan ekstensi **Polyglot Notebooks** (atau Jupyter dengan .NET Interactive) lalu jalankan sel secara berurutan. [English version](./03-ethereum-evm.en.ipynb)

Alamat EIP-55, pembacaan live dari Sepolia, kontrak ERC-20, transaksi EIP-1559, tanda tangan EIP-191 dan EIP-712, serta codec ABI.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.0.0"

In [ ]:
using System.Numerics;
using Crypto.Net.Core;
using Crypto.Net.Wallet;
using Crypto.Net.Evm;

var wallet  = HdWallet.Generate();
var account = wallet.GetEvmAccount(0, EvmChain.Sepolia);   // m/44'/60'/0'/0/0, like MetaMask
account.Address.Value

## Alamat (EIP-55)

Alamat huruf campuran membawa checksum; satu huruf yang salah langsung terdeteksi.

In [ ]:
string good = "0x5aAeb6053F3E94C9b9A09f33669435E7Ef1BeAed";
string bad  = "0x5aAeb6053F3E94C9b9A09f33669435E7Ef1BeAeD";
Console.WriteLine($"{good} valid: {EvmAddress.IsValid(good)}");
Console.WriteLine($"{bad} valid: {EvmAddress.IsValid(bad)}");
Console.WriteLine($"checksummed: {EvmAddress.ToChecksumAddress(good.ToLowerInvariant())}");

## Data live dari Sepolia

In [ ]:
var client = new EvmRpcClient(EvmChain.Sepolia);
Console.WriteLine($"chain id {await client.GetChainIdAsync()}, block {await client.GetBlockNumberAsync():N0}");
var fees = await client.EstimateFeesAsync();
Console.WriteLine($"base fee {Amount.FromWei(fees.BaseFee).WithDecimals(18)} ETH, suggested max {Amount.FromWei(fees.MaxFeePerGas)} ETH per gas");
Console.WriteLine($"your balance: {await client.GetBalanceAsync(account.Address)} ETH");

## ERC-20

`Erc20Client` membaca metadata dan menskalakan saldo dengan `decimals()`. Ini WETH di Sepolia.

In [ ]:
var weth = new Erc20Client(client, "0xfFf9976782d46CC05630D1f6eBAb18b2324d6B14");
Console.WriteLine($"{await weth.GetNameAsync()} ({await weth.GetSymbolAsync()}), {await weth.GetDecimalsAsync()} decimals");
Console.WriteLine($"total supply {await weth.GetTotalSupplyAsync()}");

## Transaksi EIP-1559

Dibangun dan ditandatangani offline, lalu di-decode lagi untuk memulihkan pengirimnya.

In [ ]:
var tx = new EvmTransaction
{
    ChainId = EvmChain.Sepolia.NumericChainId,
    Nonce = 0,
    MaxPriorityFeePerGas = 1_000_000_000,
    MaxFeePerGas = 20_000_000_000,
    GasLimit = 21_000,
    To = "0xfFf9976782d46CC05630D1f6eBAb18b2324d6B14",
    Value = Amount.FromEther(0.001m).BaseUnits,
};
var signed  = account.SignTransaction(tx);
var decoded = EvmTransaction.DecodeSigned(signed.RawBytes);

Console.WriteLine($"hash   {signed.Hash}");
Console.WriteLine($"raw    {signed.RawHex}");
Console.WriteLine($"type   {decoded.Transaction.Type}, nonce {decoded.Transaction.Nonce}, value {Amount.FromWei(decoded.Transaction.Value)} ETH");
Console.WriteLine($"sender {decoded.RecoverSender()} (ours: {decoded.RecoverSender() == account.Address.Value})");

## Tanda tangan pesan

EIP-191 `personal_sign` ("Sign in with Ethereum") dan typed data EIP-712.

In [ ]:
byte[] sig = account.SignMessage("Sign in to Crypto.Net");
Console.WriteLine($"signature {HexUtil.Encode(sig)}");
Console.WriteLine($"recovered {EvmMessageSigner.RecoverPersonalMessageSigner("Sign in to Crypto.Net", HexUtil.Encode(sig))}");

string typedData = """
{
  "types": {
    "EIP712Domain": [ {"name":"name","type":"string"}, {"name":"version","type":"string"}, {"name":"chainId","type":"uint256"} ],
    "Order": [ {"name":"maker","type":"address"}, {"name":"amount","type":"uint256"}, {"name":"note","type":"string"} ]
  },
  "primaryType": "Order",
  "domain": { "name": "Crypto.Net Demo", "version": "1", "chainId": 11155111 },
  "message": { "maker": "0x5aAeb6053F3E94C9b9A09f33669435E7Ef1BeAed", "amount": "1000000", "note": "hello" }
}
""";
byte[] typedSig = account.SignTypedData(typedData);
Console.WriteLine($"EIP-712 digest {HexUtil.Encode(EvmMessageSigner.HashTypedData(typedData))}");
Console.WriteLine($"EIP-712 signer {EvmMessageSigner.RecoverTypedDataSigner(typedData, typedSig)}");

## ABI

Mengodekan panggilan dari signature, men-decode data balikan dan log.

In [ ]:
byte[] call = EvmAbi.EncodeFunctionCall("transfer(address to, uint256 amount)", account.Address.Value, new BigInteger(1_000_000));
Console.WriteLine(HexUtil.Encode(call));

byte[] encoded = EvmAbi.Encode("string,uint256[],(address,bool)", "Crypto.Net", new object[] { 1, 2, 3 }, new object[] { account.Address.Value, true });
var values = EvmAbi.Decode("string,uint256[],(address,bool)", encoded);
Console.WriteLine($"{values[0]} · [{string.Join(", ", (object?[])values[1]!)}] · {string.Join(", ", (object?[])values[2]!)}");
Console.WriteLine($"Transfer topic {Erc20Client.TransferEventTopic}");

## Mengirim sungguhan

Dengan ETH Sepolia di akun:

```csharp
await using var signer = account.CreateSigner();
TxHash hash = await client.TransferAsync(signer, "0x…", Amount.FromEther(0.001m)); // nonce, fee, gas, simulasi
TxReceipt receipt = await client.WaitForReceiptAsync(hash);
```

In [ ]:
account.Dispose(); client.Dispose(); wallet.Dispose();
"disposed"